# DAMICORE artifacts: violence against women

This notebook is the only database-facing preparation stage for this hypothesis.
It creates the three corpora and the metadata contract consumed by the experiment
notebooks. It does not execute DAMICORE or interpret clusters.

**Scope:** distinct reports with a female victim registered from January 2020 through
June 2026, restricted to the exploratory violence-related source taxonomy.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hypotheses.violence_against_women.scripts.experiment_common import (
    BIAS_WARNING_THRESHOLD,
    CASE_BALANCED_WORK_ROOT,
    CASE_FULL_WORK_ROOT,
    COMMON_WORK_ROOT,
    END_DATE,
    HYPOTHESIS_ROOT,
    MIN_REPORT_COUNT,
    NORMALIZED_WORK_ROOT,
    RESULTS_ROOT,
    START_DATE,
    VICTIM_GENDER,
    category_order_from_manifest,
    load_artifact_manifest,
    load_category_map,
    plot_bias_diagnostics,
    plot_cluster_stability,
    plot_ncd_heatmap,
    plot_support,
    render_tree_artifacts,
    same_cluster_pairs,
    write_json,
)

load_dotenv(PROJECT_ROOT / ".env")

import json
import shutil
from math import log2

import numpy as np
import psycopg

from hypotheses.violence_against_women.scripts.damicore_case_experiment import (
    CASE_COLUMNS,
    CASE_FIELDS,
    build_case_corpora,
    compare_combination_distributions,
    load_case_records,
)
from hypotheses.violence_against_women.scripts.experiment_common import (
    ARTIFACT_ROOT,
    ARTIFACT_SCHEMA_VERSION,
    CASE_SEEDS,
    LOG_RATIO_LIMIT,
    RESULTS_ROOT,
    SMOOTHING_ALPHA,
    WORK_ROOT,
    ensure_artifact_directories,
    write_json,
)

DATABASE_URL = os.getenv(
    "DISQUE100_DATABASE_URL",
    "postgresql://postgres@127.0.0.1:5433/disque100",
)

# This hypothesis owns its generated workspace. Re-running this cell starts its work
# artifacts from zero without touching shared raw data or the database.
if ARTIFACT_ROOT.exists():
    shutil.rmtree(ARTIFACT_ROOT)
ensure_artifact_directories()


## 1. Query scope and shared context counts

`source_hash` is used only inside PostgreSQL for distinct-report counts. It is not
written to the derived artifacts.


In [ ]:
CATEGORY_SQL = """
nullif(concat_ws(' > ',
    nullif(trim(split_part(violation, '>', 1)), ''),
    nullif(trim(split_part(violation, '>', 2)), '')
), '')
"""

CONTEXT_FIELDS = [
    ("faixa_etaria", "victim_age_group"),
    ("relacao_vitima_suspeito", "victim_suspect_relationship"),
    ("ambiente", "violation_setting"),
    ("mes", "registered_at"),
    ("inicio_violacoes", "violation_start_period"),
    ("canal_atendimento", "service_channel"),
    ("tipo_denunciante", "reporter_type"),
    ("frequencia", "frequency"),
    ("situacao_emergencia", "emergency_status"),
    ("motivacao", "motivation"),
    ("grupo_vulneravel", "vulnerable_group"),
    ("deficiencia_vitima", "victim_disability"),
    ("raca_cor_vitima", "victim_race_color"),
    ("escolaridade_vitima", "victim_education_level"),
    ("renda_vitima", "victim_income_range"),
    ("etnia_vitima", "victim_ethnicity"),
    ("faixa_etaria_suspeito", "suspect_age_group"),
    ("genero_suspeito", "suspect_gender"),
    ("escolaridade_suspeito", "suspect_education_level"),
    ("natureza_juridica_suspeito", "suspect_legal_nature"),
]
assert {label for label, _ in CONTEXT_FIELDS} == {label for _, label in CASE_FIELDS}
assert len(CONTEXT_FIELDS) == len(CASE_COLUMNS) == 20

value_selects = ["source_hash", f"{CATEGORY_SQL} AS category"]
for label, source_column in CONTEXT_FIELDS:
    if label == "mes":
        expression = "to_char(date_trunc('month', registered_at), 'YYYY-MM')"
    else:
        expression = f"coalesce(nullif(trim({source_column}), ''), 'DESCONHECIDO')"
    value_selects.append(f"{expression} AS {label}")

context_selects = [
    "SELECT category, source_hash, 'denuncia'::text AS dimension, 'TODAS'::text AS value\nFROM report_values WHERE category IS NOT NULL"
]
for label, _ in CONTEXT_FIELDS:
    context_selects.append(
        f"SELECT category, source_hash, '{label}' AS dimension, {label} AS value\n"
        "FROM report_values WHERE category IS NOT NULL"
    )

coverage_query = f"""
WITH reports AS (
    SELECT source_hash,
           bool_or(violation IS NOT NULL) AS has_violation,
           bool_or(({CATEGORY_SQL}) IS NOT NULL) AS has_category
    FROM public.disque100_reports
    WHERE registered_at >= %s AND registered_at < %s
      AND victim_gender = %s
    GROUP BY source_hash
)
SELECT count(*) AS female_reports,
       count(*) FILTER (WHERE has_category) AS reports_with_category,
       count(*) FILTER (WHERE NOT has_violation) AS reports_without_violation
FROM reports
"""

context_query = f"""
WITH report_values AS (
    SELECT {', '.join(value_selects)}
    FROM public.disque100_reports
    WHERE registered_at >= %s AND registered_at < %s
      AND victim_gender = %s
), contexts AS (
    {' UNION ALL '.join(context_selects)}
)
SELECT category, dimension, value, count(DISTINCT source_hash) AS report_count
FROM contexts
GROUP BY category, dimension, value
ORDER BY category, dimension, value
"""

parameters = (START_DATE, END_DATE, VICTIM_GENDER)
with psycopg.connect(DATABASE_URL) as connection:
    with connection.cursor() as cursor:
        cursor.execute(coverage_query, parameters)
        coverage = pd.DataFrame(
            cursor.fetchall(),
            columns=[column.name for column in cursor.description],
        )
        cursor.execute(context_query, parameters)
        context_counts = pd.DataFrame(
            cursor.fetchall(),
            columns=[column.name for column in cursor.description],
        )

assert set(context_counts.columns) == {"category", "dimension", "value", "report_count"}
assert set(context_counts["dimension"].unique()) == {"denuncia", *{label for label, _ in CASE_FIELDS}}
coverage.to_csv(COMMON_WORK_ROOT / "coverage.csv", index=False)
context_counts.to_csv(COMMON_WORK_ROOT / "context-counts.csv", index=False)
display(coverage)
display(context_counts.head())
print(f"Context rows: {len(context_counts):,}")


## 2. Select eligible categories and create the normalized corpus

The selection rule follows the source taxonomy and the minimum support threshold used
by the original experiment. Category order is fixed here and reused by every notebook.


In [ ]:
category_summary = context_counts.loc[
    context_counts["dimension"] == "denuncia",
    ["category", "report_count"],
].rename(columns={"report_count": "support"})

scope_mask = (
    category_summary["category"].str.startswith(
        ("INTEGRIDADE", "VIDA", "VIOLÊNCIA INSTITUCIONAL")
    )
    | category_summary["category"].eq("LIBERDADE > SEXUAL")
    | category_summary["category"].str.contains(
        "VIOLÊNCIA POLITÍCA DE GÊNERO E CONTRA AS MULHERES",
        regex=False,
    )
)
category_support = category_summary.copy()
category_support["status"] = "included"
category_support.loc[~scope_mask, "status"] = "out_of_scope"
category_support.loc[
    scope_mask & category_support["support"].lt(MIN_REPORT_COUNT),
    "status",
] = "below_minimum_support"

included_support = category_support.loc[
    category_support["status"] == "included"
].copy()
included_support = included_support.sort_values("category").reset_index(drop=True)
category_order = included_support["category"].tolist()
assert category_order

profile_counts = context_counts.loc[
    context_counts["category"].isin(category_order)
    & context_counts["dimension"].ne("denuncia")
].copy()
vocabulary = (
    profile_counts[["dimension", "value"]]
    .drop_duplicates()
    .sort_values(["dimension", "value"])
)
expected_dimensions = {label for _, label in CASE_FIELDS}
assert set(vocabulary["dimension"]) == expected_dimensions

normalized_profiles = (
    included_support[["category", "support"]]
    .merge(vocabulary, how="cross")
    .merge(profile_counts, on=["category", "dimension", "value"], how="left")
    .sort_values(["category", "dimension", "value"])
)
normalized_profiles["report_count"] = normalized_profiles["report_count"].fillna(0).astype(int)
global_counts = (
    profile_counts.groupby(["dimension", "value"], as_index=False)["report_count"]
    .sum()
    .rename(columns={"report_count": "global_count"})
)
global_counts["global_prevalence"] = (
    global_counts["global_count"]
    / global_counts.groupby("dimension")["global_count"].transform("sum")
)
normalized_profiles = normalized_profiles.merge(
    global_counts[["dimension", "value", "global_prevalence"]],
    on=["dimension", "value"],
    how="left",
    validate="many_to_one",
)
normalized_profiles["smoothed_prevalence"] = (
    normalized_profiles["report_count"]
    + SMOOTHING_ALPHA * normalized_profiles["global_prevalence"]
) / (normalized_profiles["support"] + SMOOTHING_ALPHA)
normalized_profiles["relative_bps"] = (
    10_000
    * (
        normalized_profiles["smoothed_prevalence"].div(
            normalized_profiles["global_prevalence"]
        ).map(log2).clip(-LOG_RATIO_LIMIT, LOG_RATIO_LIMIT)
        + LOG_RATIO_LIMIT
    )
    / (2 * LOG_RATIO_LIMIT)
).round().astype(int)

category_rows = []
for number, category in enumerate(category_order, start=1):
    label = f"category-{number:03d}.txt"
    rows = normalized_profiles.loc[normalized_profiles["category"] == category]
    lines = [
        f"{row.dimension}|{row.value}|{row.relative_bps:05d}"
        for row in rows.itertuples(index=False)
    ]
    corpus_path = NORMALIZED_WORK_ROOT / "corpus" / label
    corpus_path.parent.mkdir(parents=True, exist_ok=True)
    corpus_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    category_rows.append(
        {"label": label, "category": category, "support": int(rows["support"].iloc[0])}
    )

category_map = pd.DataFrame(category_rows)
category_map.to_csv(COMMON_WORK_ROOT / "category-map.csv", index=False)
included_support.to_csv(COMMON_WORK_ROOT / "category-support.csv", index=False)
category_support.to_csv(COMMON_WORK_ROOT / "category-scope.csv", index=False)
context_counts.loc[
    context_counts["category"].isin(category_order)
].to_csv(COMMON_WORK_ROOT / "eligible-context-counts.csv", index=False)

corpus_sizes = pd.Series(
    {
        path.name: path.stat().st_size
        for path in (NORMALIZED_WORK_ROOT / "corpus").glob("*.txt")
    },
    name="bytes",
)
assert corpus_sizes.nunique() == 1
display(category_map)
print(f"Included categories: {len(category_order)}")
print(f"Normalized corpus bytes per category: {int(corpus_sizes.iloc[0]):,}")


## 3. Create the case-full and case-balanced corpora

The case grain is `source_hash + category` in memory. Canonical case documents contain
only the 20 contextual dimensions, so report identifiers are not exported.


In [ ]:
case_records = load_case_records(
    database_url=DATABASE_URL,
    start_date=START_DATE,
    end_date=END_DATE,
    victim_gender=VICTIM_GENDER,
    included_categories=category_order,
    category_sql=CATEGORY_SQL,
)
case_support = (
    case_records.groupby("category", as_index=False)["source_hash"]
    .nunique()
    .rename(columns={"source_hash": "case_count"})
)
expected_support = included_support[["category", "support"]].rename(
    columns={"support": "expected_case_count"}
)
case_support_check = expected_support.merge(
    case_support, on="category", validate="one_to_one"
).set_index("category").loc[category_order]
assert case_support_check["expected_case_count"].equals(
    case_support_check["case_count"]
)

case_category_map, case_combination_counts, balanced_sample_size = build_case_corpora(
    case_records=case_records,
    category_map=category_map,
    work_dir=WORK_ROOT,
    seeds=CASE_SEEDS,
    metadata_dir=COMMON_WORK_ROOT,
)
case_combination_comparison = compare_combination_distributions(
    case_combination_counts,
    COMMON_WORK_ROOT / "case-combination-comparison.csv",
)
case_category_map.to_csv(COMMON_WORK_ROOT / "case-category-map.csv", index=False)
case_combination_counts.to_csv(
    COMMON_WORK_ROOT / "case-combination-counts.csv", index=False
)

assert not case_records.duplicated(["source_hash", "category"]).any()
canonical_keys = case_records["canonical_record"].map(lambda value: set(json.loads(value)))
assert canonical_keys.map(lambda keys: not {"source_hash", "category", "id"}.intersection(keys)).all()
assert canonical_keys.map(lambda keys: keys == expected_dimensions).all()
assert case_category_map["case_count"].gt(0).all()
assert set(case_category_map["category"]) == set(category_order)
assert case_category_map.loc[
    case_category_map["regime"] == "case-balanced", "case_count"
].eq(balanced_sample_size).all()

print(f"Case records in memory: {len(case_records):,}")
print(f"Balanced sample size per category and replica: {balanced_sample_size:,}")
print(f"Balanced replicas: {len(CASE_SEEDS)}")
display(case_category_map.head())

# Keep identifiers and large in-memory case records out of later notebook state.
del case_records, case_combination_counts, case_combination_comparison


## 4. Write the artifact manifest

The manifest is the compatibility contract for the three experiment notebooks.


In [ ]:
manifest = {
    "schema_version": ARTIFACT_SCHEMA_VERSION,
    "hypothesis": "violence_against_women",
    "source_table": "public.disque100_reports",
    "start_date": START_DATE,
    "end_date": END_DATE,
    "victim_gender": VICTIM_GENDER,
    "minimum_report_count": MIN_REPORT_COUNT,
    "smoothing_alpha": SMOOTHING_ALPHA,
    "log_ratio_limit": LOG_RATIO_LIMIT,
    "case_seeds": CASE_SEEDS,
    "dimensions": [label for _, label in CASE_FIELDS],
    "category_order": category_order,
    "category_count": len(category_order),
    "balanced_sample_size": balanced_sample_size,
    "paths": {
        "category_map": "common/category-map.csv",
        "category_support": "common/category-support.csv",
        "case_category_map": "common/case-category-map.csv",
        "normalized_corpus": "normalized_categories/corpus",
        "case_full_corpus": "case_full/corpus",
        "case_balanced_root": "case_balanced",
    },
}
write_json(COMMON_WORK_ROOT / "artifact-manifest.json", manifest)
write_json(HYPOTHESIS_ROOT / "manifest.json", {
    "hypothesis": "violence_against_women",
    "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
    "notebooks": [
        "00_create_artifacts.ipynb",
        "01_experiment_normalized_categories.ipynb",
        "02_experiment_case_full.ipynb",
        "03_experiment_case_balanced.ipynb",
        "04_compare_experiments.ipynb",
    ],
})
print("Artifact manifest written.")
